# Surgical Scene CV – training on Google Colab
Runtime → Change runtime type → **GPU (T4)**. Full training takes roughly 30–60 min.

In [ ]:
!git clone https://github.com/irmakkoseoglu/surgical-scene-cv.git
%cd surgical-scene-cv
!pip -q install onnx

## 1. Download CholecSeg8k from Kaggle
Upload your `kaggle.json` (Kaggle → Settings → API → *Create New Token*) when asked.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d newslab/cholecseg8k -p /content/cholecseg8k --unzip -q
!ls /content/cholecseg8k | head

## 2. Check the mask encoding

In [ ]:
!python python/inspect_masks.py /content/cholecseg8k

## 3. Train (videos 12, 20, 48, 55 are held out → no frame leakage between train and validation)

In [ ]:
!python python/train.py --data /content/cholecseg8k --epochs 30 --batch 16 --out runs/unet_r34

## 4. Evaluate + export to ONNX

In [ ]:
!python python/evaluate.py --data /content/cholecseg8k --ckpt runs/unet_r34/best.pt --out results
!python python/export_onnx.py --ckpt runs/unet_r34/best.pt --out models/unet.onnx

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/samples/*.png'))[:3]:
    display(Image(f))

## 5. C++ inference with OpenCV DNN (CPU latency)

In [ ]:
!apt-get -qq install -y libopencv-dev > /dev/null
!cmake -S cpp -B cpp/build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build cpp/build -j2 > /dev/null
frame = sorted(glob.glob('/content/cholecseg8k/**/video12_*/*_endo.png', recursive=True))[0]
!./cpp/build/segment --model models/unet.onnx --input "{frame}" --output results/cpp_overlay.png
display(Image('results/cpp_overlay.png'))

## 6. Save results to your Drive / download
Commit `results/metrics.md` and 2–3 images from `results/samples/` to the repo and fill in the results table in the README.

In [ ]:
!zip -qr results.zip results models/unet.onnx runs/unet_r34/history.json
files.download('results.zip')